In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR

from xgboost import XGBRegressor


In [ ]:
path = "path/to/HanHoJu_2.xlsx"

df = pd.read_excel(path, sheet_name=0)
print(df.shape)
df.head()


In [ ]:
drop_cols = [
    "Ref_ID",
    "JV_reverse_scan_Voc",
    "JV_reverse_scan_Jsc",
    "JV_reverse_scan_FF"
]

drop_cols = [c for c in drop_cols if c in df.columns]

df_ = df.drop(columns=drop_cols)

target_col = "JV_reverse_scan_PCE"
assert target_col in df_.columns

y = df_[target_col]
X = df_.drop(columns=[target_col])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("feature columns:", X.columns.tolist()[:10], "...")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [ ]:
Xy_train = X_train.copy()
Xy_train[target_col] = y_train

num_cols = Xy_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
num_cols = [c for c in num_cols if c != target_col]

print("num_cols:", num_cols)
print("Xy_train shape:", Xy_train.shape)

In [ ]:
def interpolate_global(df, n_new=200, random_state=42):
    rng = np.random.default_rng(random_state)
    df = df.reset_index(drop=True)
    n = len(df)
    new_rows = []

    for _ in range(n_new):
        i, j = rng.choice(n, size=2, replace=False)
        alpha = rng.random()

        row_i = df.loc[i]
        row_j = df.loc[j]

        new_row = row_i.copy()

        for col in num_cols + [target_col]:
            vi = row_i[col]
            vj = row_j[col]

            if pd.isna(vi) or pd.isna(vj):
                new_row[col] = vi if not pd.isna(vi) else vj
            else:
                new_row[col] = (1 - alpha) * vi + alpha * vj

        new_rows.append(new_row)

    return pd.DataFrame(new_rows)

In [ ]:

Xy_aug = interpolate_global(Xy_train, n_new=9 * len(Xy_train), random_state=42)

print("Original train:", Xy_train.shape)
print("Augmented train:", Xy_aug.shape)

Xy_train_full = pd.concat([Xy_train, Xy_aug], axis=0).reset_index(drop=True)

X_train_aug = Xy_train_full.drop(columns=[target_col])
y_train_aug = Xy_train_full[target_col]

print("combined train:", X_train_aug.shape, y_train_aug.shape)

In [ ]:
numeric_cols = X_train_aug.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X_train_aug.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model),
])

xgb_pipe.fit(X_train_aug, y_train_aug)

y_pred = xgb_pipe.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("===== XGBoost + Interpolation =====")
print(f"MSE : {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")

In [ ]:
svr_model = SVR(
    kernel="rbf",
    C=100.0,
    epsilon=0.1,
    gamma="scale"
)

svr_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", svr_model),
])

svr_pipe.fit(X_train_aug, y_train_aug)

y_pred_svr = svr_pipe.predict(X_test)

mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mse_svr)
mae_svr = mean_absolute_error(y_test, y_pred_svr)
r2_svr = r2_score(y_test, y_pred_svr)

print("===== SVR + Interpolation =====")
print(f"MSE : {mse_svr:.4f}")
print(f"RMSE: {rmse_svr:.4f}")
print(f"MAE : {mae_svr:.4f}")
print(f"R2  : {r2_svr:.4f}")